## 0. Install & Restart

In [ ]:
import os
os.chdir("/content/")
!rm -rf LLM-Annotator
!git clone https://github.com/KingArthur0205/LLM-Annotator.git
os.chdir("/content/LLM-Annotator")
!pip install -e . -q
!pip install pypdf gspread pydrive2 scikit-learn tabulate -q
os.kill(os.getpid(), 9)  # restart runtime to pick up installed package

## 1. Mount Drive & Authenticate

In [ ]:
import os, json, re
os.chdir("/content/LLM-Annotator")

from google.colab import drive, auth
drive.mount("/content/drive")
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
ga = GoogleAuth()
ga.credentials = creds
gdrive = GoogleDrive(ga)

# Load sheet IDs (extract IDs from URLs if needed)
with open("data/sheets.json") as f:
    raw_sheets = json.load(f)

def to_sheet_id(val):
    m = re.search(r"/d/([a-zA-Z0-9_-]+)", val)
    return m.group(1) if m else val

SHEETS = {k: to_sheet_id(v) for k, v in raw_sheets.items()}

DRIVE_BASE = "/content/drive/MyDrive/EduNLP"  # adjust to your Drive path

# Load API keys from secrets sheet
from llm_annotator.secrets import load_secrets
load_secrets(gc, SHEETS["secrets"])

print("✓ Authenticated")
print(f"Sheets: {json.dumps(SHEETS, indent=2)}")

## 2. Configure Experiment

In [ ]:
from llm_annotator.ui import show_config_ui

ui = show_config_ui(drive_base=DRIVE_BASE, gc=gc, sheets=SHEETS)

## 3. Build Config & Preview

In [ ]:
config = ui.build_config()
config.tracker_sheet_id = SHEETS["tracker"]

print(f"Models:      {config.model_list}")
print(f"Features:    {config.feature_list}")
print(f"Obs list:    {config.obs_list}")
print(f"Test mode:   {config.test_mode}")
print(f"Video:       {config.use_video}")
print(f"Save dir:    {config.save_dir}")

## 4. Run Pipeline

In [ ]:
from llm_annotator.runner import run_pipeline

results = run_pipeline(
    config=config,
    results_sheet_id=SHEETS.get("results", ""),
    validation_path=ui.get_validation_path(),
    gc=gc,
    gdrive=gdrive,
    verbose=True,
    materials_folder_override=ui.get_materials_folder_override(),
)

## 5. Optional: Fetch Results (if if_wait=False)

Provide the timestamp and feature from the original run to fetch results after the fact.

In [ ]:
from llm_annotator import fetch

# Fill these in from the original run's output
feature = "Directions"
time_stamp = "2025-01-01_00:00:00"  # replace with actual timestamp

fetch(feature=feature, timestamp=time_stamp, save_dir=config.save_dir)